In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API 키가 존재하며 {google_api_key[:8]}로 시작합니다")
else:
    print("Google API 키가 설정되어 있지 않습니다")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.1-flash-lite" # 일단은 무료모델은 text-to-image 되는게 없음 다 유료
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

DB = "prices.db"

In [ ]:
system_message = """
당신은 FlightAI라는 항공사의 도움이 되는 어시스턴트입니다.
짧고 정중한 답변을 하며, 1문장을 넘기지 마세요.
항상 정확하게 답하세요. 답을 모른다면, 모른다고 말하세요.
"""

In [ ]:
def get_ticket_price(city):
    print(f"데이터베이스 도구 호출됨: {city}의 가격을 조회합니다", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"{city}로 가는 티켓 가격은 ${result[0]}입니다" if result else "이 도시에 대한 가격 정보가 없습니다"

In [ ]:
get_ticket_price("Paris")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "목적지 도시로 가는 왕복 티켓의 가격을 가져옵니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "고객이 여행하고자 하는 도시",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

# 멀티모달

In [ ]:
# 이미지 처리를 위한 몇 가지 임포트

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = gemini.images.generate(
            model="gemini-3.1-flash-image-preview", # 유료
            prompt=f"{city}에서의 휴가를 표현하는 이미지, 관광 명소와 {city}만의 독특한 모든 것을 보여주며, 화려한 팝아트 스타일로",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [ ]:
def talker(message):
    response = gemini.audio.speech.create(
      model="gemini-3.1-flash-tts-preview",
      voice="onyx",    # onyx 대신 alloy나 coral로 바꿔서 시도해보세요
      input=message
    )
    return response.content

In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])

    return history, voice, image

In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## Gradio UI의 3가지 유형

`gr.Interface`는 표준적이고 간단한 UI를 위한 것입니다

`gr.ChatInterface`는 표준 챗봇 UI를 위한 것입니다

`gr.Blocks`는 컴포넌트와 콜백을 직접 제어하는 커스텀 UI를 위한 것입니다

In [ ]:
# 콜백 (위의 chat() 함수와 함께)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI 정의

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="AI 어시스턴트와 대화해보세요:")

# 이벤트를 콜백에 연결하기

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))